# 02 - vLLM Serving

PagedAttention memory management, continuous batching, KV cache optimization, request scheduling, throughput vs latency trade-offs, deployment configuration.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import heapq
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')


## 1. PagedAttention Memory Management


In [ ]:
class PagedAttention:
    def __init__(self, page_size=16, num_pages=100):
        self.page_size = page_size
        self.num_pages = num_pages
        self.pages = [None] * num_pages
        self.page_table = {}
        self.free_pages = list(range(num_pages))
    
    def allocate(self, request_id, num_tokens):
        num_pages_needed = (num_tokens + self.page_size - 1) // self.page_size
        if len(self.free_pages) < num_pages_needed:
            return False
        allocated = []
        for _ in range(num_pages_needed):
            page_idx = self.free_pages.pop(0)
            self.pages[page_idx] = request_id
            allocated.append(page_idx)
        self.page_table[request_id] = allocated
        return True
    
    def deallocate(self, request_id):
        if request_id in self.page_table:
            for page_idx in self.page_table[request_id]:
                self.pages[page_idx] = None
                self.free_pages.append(page_idx)
            del self.page_table[request_id]
    
    def get_utilization(self):
        used = sum(1 for p in self.pages if p is not None)
        return used / self.num_pages
    
    def visualize(self):
        plt.figure(figsize=(12, 2))
        colors = ['lightgreen' if p is None else 'steelblue' for p in self.pages]
        plt.bar(range(self.num_pages), [1]*self.num_pages, color=colors, edgecolor='white', linewidth=0.5)
        plt.xlabel('Page Index')
        plt.title(f'PagedAttention (Utilization: {self.get_utilization()*100:.1f}%)')
        plt.yticks([])
        plt.tight_layout()
        plt.show()

paged_attn = PagedAttention(page_size=16, num_pages=50)
requests = [('req_1', 25), ('req_2', 40), ('req_3', 15), ('req_4', 60)]
for req_id, tokens in requests:
    success = paged_attn.allocate(req_id, tokens)
    print(f'{req_id} ({tokens} tokens): Allocated = {success}')
print(f'Utilization: {paged_attn.get_utilization()*100:.1f}%')
paged_attn.visualize()


## 2. KV Cache Management


In [ ]:
class KVCache:
    def __init__(self, max_cache_size=1000):
        self.cache = {}
        self.max_size = max_cache_size
        self.current_size = 0
    
    def store(self, request_id, layer_id, k, v):
        if request_id not in self.cache:
            self.cache[request_id] = {}
        self.cache[request_id][layer_id] = (k, v)
        self.current_size += k.size + v.size
    
    def get(self, request_id, layer_id):
        if request_id in self.cache and layer_id in self.cache[request_id]:
            return self.cache[request_id][layer_id]
        return None
    
    def evict_oldest(self, n=1):
        for _ in range(n):
            if self.cache:
                oldest = next(iter(self.cache))
                del self.cache[oldest]
    
    def get_stats(self):
        total_requests = len(self.cache)
        total_layers = sum(len(layers) for layers in self.cache.values())
        size_mb = self.current_size / (1024**2)
        return {'requests': total_requests, 'layers': total_layers, 'size_mb': size_mb}

kv_cache = KVCache(max_cache_size=10000)
for req_id in ['req_1', 'req_2', 'req_3']:
    for layer in range(12):
        k = np.random.randn(1, 32, 128).astype(np.float16)
        v = np.random.randn(1, 32, 128).astype(np.float16)
        kv_cache.store(req_id, layer, k, v)
stats = kv_cache.get_stats()
print(f'KV Cache Stats: {stats}')


## 3. Continuous Batching


In [ ]:
class ContinuousBatchingScheduler:
    def __init__(self, max_batch_size=8, max_tokens_per_batch=512):
        self.max_batch_size = max_batch_size
        self.max_tokens = max_tokens_per_batch
        self.waiting_queue = deque()
        self.running_batch = []
        self.completed = []
    
    def add_request(self, request_id, prompt_len, max_new_tokens):
        self.waiting_queue.append({
            'id': request_id,
            'prompt_len': prompt_len,
            'max_new': max_new_tokens,
            'tokens_generated': 0,
            'status': 'waiting'
        })
    
    def schedule(self):
        while len(self.running_batch) < self.max_batch_size and self.waiting_queue:
            req = self.waiting_queue.popleft()
            req['status'] = 'running'
            self.running_batch.append(req)
    
    def step(self):
        completed_this_step = []
        for req in self.running_batch:
            req['tokens_generated'] += 1
            if req['tokens_generated'] >= req['max_new']:
                req['status'] = 'completed'
                completed_this_step.append(req)
        self.running_batch = [r for r in self.running_batch if r['status'] == 'running']
        self.completed.extend(completed_this_step)
        self.schedule()
        return len(completed_this_step)
    
    def get_stats(self):
        return {
            'waiting': len(self.waiting_queue),
            'running': len(self.running_batch),
            'completed': len(self.completed)
        }

scheduler = ContinuousBatchingScheduler(max_batch_size=4)
for i in range(10):
    scheduler.add_request(f'req_{i}', prompt_len=20 + i*5, max_new_tokens=5 + i*2)
print('Continuous Batching Simulation:')
for step in range(20):
    completed = scheduler.step()
    stats = scheduler.get_stats()
    print(f'Step {step+1}: running={stats["running"]}, waiting={stats["waiting"]}, completed={stats["completed"]}')
    if stats['waiting'] == 0 and stats['running'] == 0:
        break


## 4. Request Scheduling Strategies


In [ ]:
class RequestScheduler:
    def __init__(self, strategy='fcfs'):
        self.strategy = strategy
        self.requests = []
    
    def add(self, request):
        self.requests.append(request)
    
    def schedule(self, max_batch_size):
        if self.strategy == 'fcfs':
            batch = self.requests[:max_batch_size]
            self.requests = self.requests[max_batch_size:]
            return batch
        elif self.strategy == 'shortest':
            self.requests.sort(key=lambda x: x['prompt_len'] + x['max_new'])
            batch = self.requests[:max_batch_size]
            self.requests = self.requests[max_batch_size:]
            return batch
        elif self.strategy == 'priority':
            self.requests.sort(key=lambda x: x.get('priority', 0), reverse=True)
            batch = self.requests[:max_batch_size]
            self.requests = self.requests[max_batch_size:]
            return batch
        return []

strategies = ['fcfs', 'shortest', 'priority']
results = {}
for strat in strategies:
    scheduler = RequestScheduler(strategy=strat)
    for i in range(20):
        scheduler.add({
            'id': f'req_{i}',
            'prompt_len': np.random.randint(10, 100),
            'max_new': np.random.randint(5, 50),
            'priority': np.random.randint(1, 10)
        })
    total_latency = 0
    batch_count = 0
    while scheduler.requests:
        batch = scheduler.schedule(4)
        batch_latency = max(r['prompt_len'] + r['max_new'] for r in batch) if batch else 0
        total_latency += batch_latency
        batch_count += 1
    results[strat] = {'total_latency': total_latency, 'batches': batch_count}

print('Scheduling Strategy Comparison:')
for strat, res in results.items():
    print(f'  {strat.upper()}: Total Latency={res["total_latency"]}, Batches={res["batches"]}')


## 5. Throughput vs Latency Trade-offs


In [ ]:
batch_sizes = [1, 2, 4, 8, 16, 32]
throughput = []
latency = []
for bs in batch_sizes:
    tp = bs * 50 / (1 + 0.1 * bs)
    lat = 50 + 5 * bs
    throughput.append(tp)
    latency.append(lat)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(batch_sizes, throughput, 'b-o', linewidth=2, markersize=8)
ax1.set_xlabel('Batch Size')
ax1.set_ylabel('Throughput (tokens/sec)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax2 = ax1.twinx()
ax2.plot(batch_sizes, latency, 'r-s', linewidth=2, markersize=8)
ax2.set_ylabel('Latency (ms)', color='red')
ax2.tick_params(axis='y', labelcolor='red')
plt.title('Throughput vs Latency Trade-off')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Deployment Configuration


In [ ]:
vllm_config = {
    'model': 'meta-llama/Llama-2-7b-hf',
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'max_num_seqs': 256,
    'max_model_len': 4096,
    'dtype': 'float16',
    'quantization': None,
    'swap_space': 4,
    'enforce_eager': False,
    'max_num_batched_tokens': 4096
}

print('vLLM Deployment Configuration:')
for key, value in vllm_config.items():
    print(f'  {key}: {value}')

def estimate_resources(model_params, dtype_bits, gpu_utilization=0.9):
    model_size_gb = model_params * dtype_bits / 8 / 1e9
    kv_cache_gb = 2
    total_needed = model_size_gb + kv_cache_gb
    gpu_memory_needed = total_needed / gpu_utilization
    return {'model_size_gb': model_size_gb, 'total_needed_gb': total_needed, 'gpu_memory_gb': gpu_memory_needed}

resources = estimate_resources(7e9, 16)
print(f'\nResource Estimation for 7B FP16 model:')
for k, v in resources.items():
    print(f'  {k}: {v:.2f} GB')


## 7. Exercises


In [ ]:
# Exercise 1: Implement speculative decoding
# Exercise 2: Add prefix caching optimization
# Exercise 3: Simulate multi-GPU tensor parallelism
# Exercise 4: Build a load balancer for multiple vLLM instances
print('Exercises listed!')
